# Priority 3 — Statistical Significance Tests

Runs Wilcoxon signed-rank tests and Friedman tests to determine whether Deep-MELU's performance differences vs baselines are statistically significant across datasets.

**Required by Pattern Recognition reviewers since 2022.**

Tests performed:
- Wilcoxon signed-rank (pairwise: Deep-MELU vs each baseline)
- Friedman test (all methods simultaneously)
- Nemenyi post-hoc test (critical difference diagram)
- Effect size (Cohen's d equivalent for paired data)

> **Run notebook 1 first** to generate `outputs/adbench_results_full.csv`  
> or the synthetic fallback below will be used automatically.

## Cell 1 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from scipy.stats import wilcoxon, friedmanchisquare, rankdata
from itertools import combinations
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

# Optional: scikit-posthocs for Nemenyi test
try:
    import scikit_posthocs as sp
    HAS_POSTHOC = True
    print("scikit-posthocs available — Nemenyi test enabled")
except ImportError:
    HAS_POSTHOC = False
    print("scikit-posthocs not found — will use manual Nemenyi approximation")
    print("Install with: pip install scikit-posthocs")

print("Imports OK")


## Cell 2 — Load results

Loads from CSV if notebook 1 was run, otherwise uses synthetic results that mirror realistic Deep-MELU performance across 8 datasets.

In [ ]:
import os

if os.path.exists("outputs/adbench_results_full.csv"):
    df = pd.read_csv("outputs/adbench_results_full.csv")
    DATASETS = df["Dataset"].unique().tolist()
    METHODS  = df["Method"].unique().tolist()
    print(f"Loaded real results: {len(DATASETS)} datasets, {len(METHODS)} methods")
    print(f"Datasets: {DATASETS}")
else:
    print("Real results not found — using synthetic fallback for demonstration")
    print("Run notebook 1 first for real ADBench results.\n")
    # Synthetic results that mirror realistic performance gaps
    DATASETS = ["Thyroid","WBC","Annthyroid","Lympho",
                "Cardiotocography","Ionosphere","Arrhythmia","Satellite"]
    METHODS  = ["Deep-MELU","Isolation Forest","LOF","OC-SVM","Vanilla AE"]
    np.random.seed(42)
    # Base AUROC profiles per dataset (realistic from literature + our benchmarks)
    base = {
        "Thyroid":          [0.946,0.932,0.941,0.930,0.890],
        "WBC":              [0.978,0.961,0.972,0.970,0.942],
        "Annthyroid":       [0.823,0.800,0.832,0.801,0.740],
        "Lympho":           [0.995,0.981,0.990,0.988,0.951],
        "Cardiotocography": [0.712,0.689,0.715,0.698,0.634],
        "Ionosphere":       [0.883,0.856,0.891,0.876,0.812],
        "Arrhythmia":       [0.794,0.755,0.769,0.762,0.710],
        "Satellite":        [0.651,0.638,0.653,0.641,0.580],
    }
    rows = []
    for ds in DATASETS:
        for i, m in enumerate(METHODS):
            noise = np.random.randn() * 0.012
            auroc = np.clip(base[ds][i] + noise, 0.5, 1.0)
            aucpr = np.clip(auroc - 0.05 + np.random.randn()*0.01, 0.3, 1.0)
            rows.append({"Dataset":ds,"Method":m,
                         "AUROC":round(auroc,4),"AUCPR":round(aucpr,4)})
    df = pd.DataFrame(rows)

# Build per-method AUROC arrays aligned by dataset
auroc_by_method = {}
for method in METHODS:
    auroc_by_method[method] = df[df["Method"]==method].set_index("Dataset")["AUROC"]
    auroc_by_method[method] = auroc_by_method[method].reindex(DATASETS).values

print(f"\nDatasets ({len(DATASETS)}): {DATASETS}")
print(f"Methods  ({len(METHODS)}): {METHODS}")


## Cell 3 — Wilcoxon signed-rank tests

Tests whether Deep-MELU's AUROC is significantly different from each baseline.  
**H₀:** no systematic difference in AUROC across datasets  
**H₁:** systematic difference exists  
Significance level: α = 0.05  
Requires ≥ 6 datasets for reliable results.

In [ ]:
dm_scores = auroc_by_method["Deep-MELU"]
baselines = [m for m in METHODS if m != "Deep-MELU"]

print("Wilcoxon Signed-Rank Test: Deep-MELU vs each baseline")
print(f"Number of datasets: {len(DATASETS)}")
print(f"Significance level: α = 0.05")
print("="*65)
print(f"{'Baseline':<25}  {'W stat':>7}  {'p-value':>9}  "
      f"{'sig?':>5}  {'direction':>12}  {'effect r':>9}")
print("-"*65)

wilcoxon_results = {}
for baseline in baselines:
    bl_scores = auroc_by_method[baseline]
    diffs = dm_scores - bl_scores

    # Handle zero differences (required for Wilcoxon)
    nonzero = diffs[diffs != 0]
    if len(nonzero) < 3:
        print(f"{baseline:<25}  {'N/A':>7}  {'N/A':>9}  {'N/A':>5}  "
              f"{'insufficient data':>12}")
        continue

    try:
        stat, pval = wilcoxon(dm_scores, bl_scores, alternative="two-sided")
    except ValueError:
        # All differences zero
        stat, pval = 0., 1.0

    sig   = "✓ YES" if pval < 0.05 else "  no"
    direc = "DM better" if diffs.mean() > 0 else "DM worse"

    # Effect size r = Z / sqrt(n) (matched-pairs)
    n = len(diffs)
    z = stats.norm.ppf(min(pval/2, 1-1e-10))
    r = abs(z) / np.sqrt(n)

    print(f"{baseline:<25}  {stat:>7.1f}  {pval:>9.4f}  "
          f"{sig:>5}  {direc:>12}  {r:>9.3f}")
    wilcoxon_results[baseline] = dict(stat=stat, pval=pval,
                                      sig=(pval<0.05), direction=direc, r=r)

print("-"*65)
print("Effect size r: 0.1=small, 0.3=medium, 0.5=large")


## Cell 4 — Friedman test

Non-parametric one-way repeated measures test across all methods simultaneously.  
**H₀:** all methods perform equally across datasets  
Tests whether at least one method differs significantly from the others.

In [ ]:
# Build score matrix [n_datasets x n_methods]
score_matrix = np.column_stack([auroc_by_method[m] for m in METHODS])

friedman_stat, friedman_p = friedmanchisquare(*score_matrix.T)

print("Friedman Test — all methods simultaneously")
print("="*50)
print(f"  χ² statistic : {friedman_stat:.4f}")
print(f"  p-value      : {friedman_p:.6f}")
print(f"  Significant  : {'YES ✓' if friedman_p < 0.05 else 'no'}")
print()
if friedman_p < 0.05:
    print("  → At least one method differs significantly.")
    print("  → Proceed to post-hoc pairwise tests (Nemenyi).")
else:
    print("  → No significant overall difference.")
    print("  → Individual Wilcoxon tests still informative.")

# Average ranks per method
ranks = np.array([rankdata(-score_matrix[i]) for i in range(len(DATASETS))])
avg_ranks = ranks.mean(axis=0)
print(f"\nAverage ranks (lower = better):")
for m, r in sorted(zip(METHODS, avg_ranks), key=lambda x: x[1]):
    bar = "█" * int(r * 5)
    print(f"  {m:<25}  rank={r:.3f}  {bar}")


## Cell 5 — Nemenyi post-hoc test

In [ ]:
if HAS_POSTHOC:
    print("Nemenyi post-hoc pairwise p-values (from scikit-posthocs):")
    nemenyi = sp.posthoc_nemenyi_friedman(score_matrix)
    nemenyi.columns = METHODS
    nemenyi.index   = METHODS
    display(nemenyi.round(4).style
            .background_gradient(cmap="RdYlGn_r", vmin=0, vmax=0.1)
            .format("{:.4f}")
            .set_caption("Nemenyi p-values (< 0.05 = significant difference)"))
else:
    # Manual critical difference approximation
    print("Manual Critical Difference (CD) diagram approximation")
    print("(install scikit-posthocs for full Nemenyi matrix)\n")

    k = len(METHODS)
    n_ds = len(DATASETS)
    # CD formula for α=0.05 from Demsar 2006
    q_alpha = {5:2.728, 6:2.850, 7:2.949, 8:3.031, 9:3.102, 10:3.164}
    q = q_alpha.get(k, 2.728)
    CD = q * np.sqrt(k*(k+1) / (6*n_ds))
    print(f"  k={k} methods, n={n_ds} datasets")
    print(f"  q_0.05 = {q}")
    print(f"  Critical Difference (CD) = {CD:.4f}\n")
    print(f"  Methods with |rank_i - rank_j| > CD = {CD:.3f} differ significantly\n")
    print(f"  {'Method pair':<45}  {'|Δrank|':>8}  {'sig?':>6}")
    print("  " + "-"*62)
    for (m1, r1), (m2, r2) in combinations(zip(METHODS, avg_ranks), 2):
        diff = abs(r1 - r2)
        sig  = "✓" if diff > CD else "  "
        print(f"  {m1:<20} vs {m2:<20}  {diff:>8.3f}  {sig:>6}")


## Cell 6 — Critical difference diagram

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
fig.suptitle("Critical Difference Diagram — Pattern Recognition submission",
             fontsize=12)

# Sort by average rank
sorted_methods = sorted(zip(METHODS, avg_ranks), key=lambda x: x[1])
method_names_sorted = [x[0] for x in sorted_methods]
ranks_sorted        = [x[1] for x in sorted_methods]

# CD line
k     = len(METHODS)
n_ds  = len(DATASETS)
q_tab = {5:2.728, 6:2.850, 7:2.949, 8:3.031, 9:3.102, 10:3.164}
q     = q_tab.get(k, 2.728)
CD    = q * np.sqrt(k*(k+1) / (6*n_ds))

ax.set_xlim(0.5, k + 0.5)
ax.set_ylim(-1.5, 2.5)
ax.invert_xaxis()  # best rank on right

# Draw rank axis
ax.axhline(0, color="black", lw=1.5)
for i in range(1, k+1):
    ax.plot(i, 0, "k|", ms=10)
    ax.text(i, -0.3, str(i), ha="center", fontsize=10)
ax.text((k+1)/2, -0.7, "Average rank →", ha="center", fontsize=9,
        color="gray")

# Plot methods
COLORS_STAT = {
    "Deep-MELU":       "#1D9E75",
    "Isolation Forest":"#534AB7",
    "LOF":             "#BA7517",
    "OC-SVM":          "#888780",
    "Vanilla AE":      "#D85A30",
}
for i, (m, r) in enumerate(zip(method_names_sorted, ranks_sorted)):
    y_pos = 1.2 if i % 2 == 0 else 1.8
    col = COLORS_STAT.get(m, "#888")
    ax.plot([r, r], [0, y_pos], color=col, lw=1.5, ls="--", alpha=0.6)
    ax.plot(r, y_pos, "o", color=col, ms=9, zorder=5)
    ax.text(r, y_pos + 0.22, m, ha="center", fontsize=9,
            fontweight="bold" if "Deep-MELU" in m else "normal",
            color=col)
    ax.text(r, y_pos - 0.22, f"{r:.2f}", ha="center", fontsize=8,
            color="gray")

# CD bar
best_r = min(ranks_sorted)
ax.annotate("", xy=(best_r + CD, 2.3), xytext=(best_r, 2.3),
            arrowprops=dict(arrowstyle="<->", color="black", lw=1.5))
ax.text(best_r + CD/2, 2.42, f"CD={CD:.3f}", ha="center", fontsize=9)

ax.axis("off")
ax.set_title(f"n={n_ds} datasets, α=0.05, CD={CD:.3f}  "
             f"(Demšar 2006 framework)", fontsize=10, pad=20)

plt.tight_layout()
plt.savefig("outputs/critical_difference_diagram.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → outputs/critical_difference_diagram.png")


## Cell 7 — Effect size and summary table

Compiles all test results into the table format expected by Pattern Recognition reviewers.

In [ ]:
print("Statistical significance summary table")
print("(ready for copy-paste into LaTeX or the paper)\n")

summary_rows = []
for baseline in baselines:
    dm_s  = auroc_by_method["Deep-MELU"]
    bl_s  = auroc_by_method[baseline]
    diffs = dm_s - bl_s

    try:
        w_stat, p_val = wilcoxon(dm_s, bl_s, alternative="two-sided")
    except ValueError:
        w_stat, p_val = 0., 1.0

    # Cohen's d for paired data
    mean_diff = diffs.mean()
    std_diff  = diffs.std() + 1e-9
    cohens_d  = mean_diff / std_diff

    # Effect size r
    n = len(diffs)
    z = stats.norm.ppf(min(max(p_val/2, 1e-10), 1-1e-10))
    r = abs(z) / np.sqrt(n)
    r_label = ("large" if r >= 0.5 else
               "medium" if r >= 0.3 else "small")

    summary_rows.append({
        "Comparison":    f"Deep-MELU vs {baseline}",
        "DM avg AUROC":  round(dm_s.mean(), 4),
        "BL avg AUROC":  round(bl_s.mean(), 4),
        "Avg Δ AUROC":   round(mean_diff, 4),
        "W statistic":   round(w_stat, 1),
        "p-value":       round(p_val, 4),
        "p < 0.05":      "Yes ✓" if p_val < 0.05 else "No",
        "Effect r":      round(r, 3),
        "Effect size":   r_label,
        "Cohen's d":     round(cohens_d, 3),
    })

df_stats = pd.DataFrame(summary_rows)
display(df_stats.style
        .applymap(lambda v: "color: #085041; font-weight:bold"
                  if v == "Yes ✓" else "", subset=["p < 0.05"])
        .applymap(lambda v: "color: #085041"
                  if isinstance(v,str) and "large" in v else "", subset=["Effect size"])
        .format({"p-value": "{:.4f}", "Avg Δ AUROC": "{:+.4f}"})
        .set_caption("Wilcoxon test results — Deep-MELU vs baselines"))

df_stats.to_csv("outputs/significance_results.csv", index=False)
print("\nCSV saved → outputs/significance_results.csv")

# LaTeX snippet
print("\n--- LaTeX table snippet (copy to paper) ---\n")
print("\\begin{table}[h]")
print("\\centering")
print("\\caption{Wilcoxon signed-rank test results (Deep-MELU vs baselines, 8 ADBench datasets)}")
print("\\begin{tabular}{lrrrcrc}")
print("\\hline")
print("Comparison & DM AUROC & BL AUROC & $\\Delta$ AUROC & $W$ & $p$-value & Sig. \\\\")
print("\\hline")
for _, row in df_stats.iterrows():
    cmp  = row["Comparison"].replace("Deep-MELU vs ", "vs ")
    sig  = "$\\checkmark$" if "Yes" in str(row["p < 0.05"]) else "—"
    print(f"{cmp} & {row['DM avg AUROC']:.4f} & {row['BL avg AUROC']:.4f} "
          f"& {row['Avg Δ AUROC']:+.4f} & {row['W statistic']:.0f} "
          f"& {row['p-value']:.4f} & {sig} \\\\")
print("\\hline")
print("\\end{tabular}")
print("\\end{table}")


## Cell 8 — Visual summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Deep-MELU — Statistical Significance Summary", fontsize=13)

# Panel 1: p-values
ax = axes[0]
comparisons = [f"vs {b}" for b in baselines]
pvals = [wilcoxon(auroc_by_method["Deep-MELU"],
                   auroc_by_method[b], alternative="two-sided")[1]
          for b in baselines]
colors_p = ["#1D9E75" if p < 0.05 else "#D85A30" for p in pvals]
bars = ax.barh(comparisons, pvals, color=colors_p, alpha=0.85)
ax.axvline(0.05, color="black", lw=1.5, ls="--", label="α=0.05")
ax.axvline(0.01, color="gray",  lw=1.0, ls=":",  label="α=0.01")
ax.set_xlabel("p-value"); ax.set_title("Wilcoxon p-values")
ax.legend(fontsize=9)
for bar, p in zip(bars, pvals):
    ax.text(p + 0.002, bar.get_y()+bar.get_height()/2,
            f"{p:.4f}", va="center", fontsize=9)

# Panel 2: average rank bar
ax = axes[1]
sorted_pairs = sorted(zip(METHODS, avg_ranks), key=lambda x: x[1])
names_r  = [x[0] for x in sorted_pairs]
ranks_r  = [x[1] for x in sorted_pairs]
colors_r = [COLORS_STAT.get(m,"#888") for m in names_r]
ax.barh(names_r, ranks_r, color=colors_r, alpha=0.85)
ax.axvline(CD + min(ranks_r), color="gray", lw=1, ls=":",
           label=f"CD = {CD:.3f}")
ax.set_xlabel("Average rank (lower = better)")
ax.set_title("Friedman avg ranks")
for i, (n_, r_) in enumerate(zip(names_r, ranks_r)):
    ax.text(r_+0.02, i, f"{r_:.2f}", va="center", fontsize=9)
ax.legend(fontsize=9)

# Panel 3: AUROC delta vs baselines
ax = axes[2]
dm_avg = auroc_by_method["Deep-MELU"].mean()
deltas = [auroc_by_method["Deep-MELU"].mean() - auroc_by_method[b].mean()
          for b in baselines]
colors_d = ["#1D9E75" if d > 0 else "#D85A30" for d in deltas]
ax.barh(baselines, deltas, color=colors_d, alpha=0.85)
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("Avg AUROC difference (Deep-MELU − baseline)")
ax.set_title("Mean AUROC advantage")
for i, (b, d) in enumerate(zip(baselines, deltas)):
    ax.text(d + (0.001 if d >= 0 else -0.001), i,
            f"{d:+.4f}", va="center",
            ha="left" if d >= 0 else "right", fontsize=9)

plt.tight_layout()
plt.savefig("outputs/significance_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → outputs/significance_summary.png")
